# Tools and Tool Calling

**Tools** are functions (or Runnables) that a chat model can **invoke** at runtime. Instead of answering from memory alone, the model emits **structured tool calls** — a function name plus JSON arguments — your code executes the function and returns results via **`ToolMessage`**. That loop unlocks live APIs, databases, calculators, and RAG lookups (**11. RAG with LCEL**).

This notebook covers defining tools with **`@tool`**, **`StructuredTool`**, and **`Tool`**, reading **`AIMessage.tool_calls`**, **`bind_tools`**, the manual **tool execution loop**, multi-tool registries, **`ToolMessage`** contracts, async tools, error handling, wrapping retrieval as a tool, and testing without live API calls.

Prerequisites: **02. Runnable Protocol and LCEL**, **03. Chat Models and Messages**, **05. Output Parsers and Structured Output**. Full agent orchestration with **`create_agent`** is **13. Agents with create_agent** — here we build the primitives that agents automate.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core

print("langchain-core:", langchain_core.__version__)

---

## 1. The Tool-Calling Loop

Tool use is a **multi-message conversation**, not a single request/response:

```
User: "What is note n2 about?"
         │
         ▼
    Chat model (tools bound)
         │
         ▼
 AIMessage(tool_calls=[{name: get_note, args: {note_id: n2}}])
         │
         ▼
    Your code executes get_note("n2")
         │
         ▼
 ToolMessage(content="Run alembic upgrade head...", tool_call_id=...)
         │
         ▼
    Chat model (same tools bound)
         │
         ▼
 AIMessage(content="Note n2 says to run alembic upgrade head.")
```

| Role | Message type | Who creates it |
|------|--------------|--------------|
| User | **`HumanMessage`** | Application |
| Model chooses tool | **`AIMessage`** with **`tool_calls`** | LLM |
| Tool result | **`ToolMessage`** | Your executor code |
| Final answer | **`AIMessage`** (no tool_calls) | LLM |

Every tool is a **`BaseTool`** — also a **`Runnable`**. Agents (**13**) automate this loop; understanding the manual loop first makes agent debugging straightforward.

---

## 2. Setup — Model and Demo Data

Lesson tools operate on a tiny in-memory **Notes API** knowledge base:

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
]

---

## 3. Defining Tools with @tool

The **`@tool`** decorator wraps a Python function as a **`StructuredTool`**. The model sees:

- **`name`** — function name (or override via `@tool("custom_name")`)
- **`description`** — docstring (required unless you pass `description=`)
- **`args`** — JSON schema inferred from type hints

In [ ]:
from langchain_core.tools import tool


@tool
def get_note(note_id: str) -> str:
    """Fetch a note by id (n1, n2, n3, n4)."""
    return NOTES.get(note_id, f"Unknown note id: {note_id}")


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


print("name:", get_note.name)
print("description:", get_note.description)
print("args schema:", json.dumps(get_note.args, indent=2))

In [ ]:
print(get_note.invoke({"note_id": "n2"}))
print(add.invoke({"a": 19, "b": 21}))

### 3.1 Docstrings Matter

The model **chooses tools from descriptions**. Vague docstrings → wrong tool selection. Include:

- **What** the tool does
- **Parameter semantics** (allowed ids, units, formats)
- **When not to use it** if ambiguous vs another tool

---

## 4. StructuredTool and Tool Constructors

Not every tool starts as a decorated function:

In [ ]:
from langchain_core.tools import StructuredTool, Tool


def _search_notes(query: str) -> str:
    q = query.lower()
    hits = [c for c in DOC_CHUNKS if q in c["text"].lower() or q in c["id"]]
    if not hits:
        return "No matching notes."
    return "\n".join(f"[{c['id']}] {c['text']}" for c in hits)


search_notes = StructuredTool.from_function(
    func=_search_notes,
    name="search_notes",
    description="Search internal documentation chunks by keyword.",
)

echo = Tool(
    name="echo",
    description="Echo the input string unchanged (demo single-arg tool).",
    func=lambda x: x,
)

print(search_notes.invoke({"query": "alembic"}))
print(echo.invoke("hello tools"))

| Constructor | Best for |
|-------------|----------|
| **`@tool`** | Typed functions — default choice |
| **`StructuredTool.from_function`** | Existing functions you cannot decorate |
| **`Tool`** | Legacy single-string-argument tools |

Prefer **`@tool`** or **`StructuredTool`** — they produce rich JSON schemas the model can fill accurately.

---

## 5. Custom args_schema with Pydantic

When type hints are insufficient, pass an explicit Pydantic model:

In [ ]:
from pydantic import BaseModel, Field


class ListNotesInput(BaseModel):
    prefix: str = Field(description="Note id prefix filter, e.g. 'n' for all notes")
    limit: int = Field(default=10, ge=1, le=50, description="Max notes to return")


@tool(args_schema=ListNotesInput)
def list_notes(prefix: str, limit: int = 10) -> str:
    """List note ids and previews filtered by id prefix."""
    items = [(k, v) for k, v in NOTES.items() if k.startswith(prefix)][:limit]
    return "\n".join(f"{k}: {v[:60]}..." for k, v in items) or "No notes matched."


print(list_notes.invoke({"prefix": "n", "limit": 2}))

**Field descriptions** become part of the tool schema the model reads — treat them like prompt engineering.

---

## 6. bind_tools — Attach Tools to the Model

**`.bind_tools(tools)`** returns a new Runnable that sends tool definitions to the provider on each call (**02**, **03**):

In [ ]:
tools = [get_note, search_notes, add, list_notes]
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "Find documentation about Alembic migrations and summarize the command."
)

print("content:", response.content or "(empty — model chose tool)")
print("tool_calls:", response.tool_calls)

### 6.1 When content Is Empty

If the model decides to call a tool, **`content`** is often empty and **`tool_calls`** is populated. Do not treat empty content as an error — execute the tools and call the model again.

### 6.2 tool_choice

Optionally constrain selection:

| Value | Behavior |
|-------|----------|
| `"auto"` | Model decides (default) |
| `"none"` | No tool calls |
| `"required"` | Must call a tool |
| Specific tool name | Force one tool, e.g. `"get_note"` |

```python
llm.bind_tools(tools, tool_choice="add")  # force the add tool
```

---

## 7. Anatomy of a tool_call

Each entry in **`AIMessage.tool_calls`** is a dict:

In [ ]:
math_response = llm.bind_tools([add]).invoke("Use the add tool for 40 + 2.")

if math_response.tool_calls:
    tc = math_response.tool_calls[0]
    print(json.dumps(tc, indent=2))
    print("name:", tc["name"])
    print("args:", tc["args"])
    print("id:", tc["id"])

| Field | Purpose |
|-------|--------|
| **`name`** | Which tool to invoke |
| **`args`** | Parsed JSON arguments (dict) |
| **`id`** | Correlates with **`ToolMessage.tool_call_id`** |

Check **`invalid_tool_calls`** when the model emits malformed JSON — log and retry or ask the user to rephrase.

---

## 8. Executing a Single Tool Call

Invoke the tool with **`args`**, then append a **`ToolMessage`**:

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

if math_response.tool_calls:
    tc = math_response.tool_calls[0]
    tool_result = add.invoke(tc["args"])

    follow_up = [
        HumanMessage(content="Use the add tool for 40 + 2."),
        math_response,
        ToolMessage(content=str(tool_result), tool_call_id=tc["id"]),
    ]

    final = llm.bind_tools([add]).invoke(follow_up)
    print("tool result:", tool_result)
    print("final answer:", final.content)

**Rules for ToolMessage:**

1. **`tool_call_id`** must match the **`id`** from the corresponding **`tool_calls`** entry.
2. **`content`** is a string (or structured blocks for multimodal — see provider docs).
3. Append **after** the **`AIMessage`** that requested the call.
4. One **`ToolMessage`** per tool call when the model requests multiple tools in parallel.

---

## 9. Tool Registry and execute_tool_calls

Production code maps tool names to callables:

In [ ]:
TOOLS_BY_NAME = {t.name: t for t in tools}


def execute_tool_calls(ai_message: AIMessage) -> list[ToolMessage]:
    messages = []
    for tc in ai_message.tool_calls:
        tool = TOOLS_BY_NAME[tc["name"]]
        try:
            result = tool.invoke(tc["args"])
        except Exception as exc:
            result = f"Tool error: {exc}"
        messages.append(
            ToolMessage(content=str(result), tool_call_id=tc["id"])
        )
    return messages


probe = llm_with_tools.invoke("Use search_notes for jwt, then get_note n3 if helpful.")
print("tool_calls:", [tc["name"] for tc in probe.tool_calls])
tool_msgs = execute_tool_calls(probe)
for tm in tool_msgs:
    print(tm.content[:80], "...")

---

## 10. Manual Multi-Step Tool Loop

This is the loop **`create_agent`** (**13**) automates. Cap iterations to prevent runaway cost:

In [ ]:
def run_tool_loop(user_input: str, max_steps: int = 5) -> str:
    messages = [HumanMessage(content=user_input)]
    bound = llm.bind_tools(tools)

    for step in range(max_steps):
        ai = bound.invoke(messages)
        messages.append(ai)

        if not ai.tool_calls:
            return ai.content or ""

        messages.extend(execute_tool_calls(ai))

    return "Stopped: max tool steps reached."


print(run_tool_loop("What pytest file holds DB fixtures? Use tools."))
print("---")
print(run_tool_loop("What is 17 plus 28? Use the add tool."))

### 10.1 Loop Flow

```
messages = [HumanMessage(...)]
repeat:
    ai = llm.bind_tools(tools).invoke(messages)
    messages.append(ai)
    if ai.tool_calls empty: return ai.content
    messages.extend(ToolMessage for each call)
```

Agents add checkpointing, middleware, and structured final outputs on top of this skeleton.

---

## 11. Parallel Tool Calls

Models may request **multiple tools in one turn**. Execute all calls before the next model invocation:

In [ ]:
parallel_prompt = (
    "Call get_note for n1 AND get_note for n3 in one step if you can. "
    "Then compare API framework vs auth header."
)

messages = [HumanMessage(content=parallel_prompt)]
ai = llm_with_tools.invoke(messages)
messages.append(ai)

print(f"requested {len(ai.tool_calls)} tool call(s):", [tc["name"] for tc in ai.tool_calls])
messages.extend(execute_tool_calls(ai))

final = llm_with_tools.invoke(messages)
print(final.content)

Never call the model again after only **some** of the requested tools — the provider expects one **`ToolMessage`** per **`tool_call_id`**.

---

## 12. Async Tools

Tools can wrap async I/O (HTTP, DB). Define an **`async def`** and LangChain routes through **`ainvoke`**:

In [ ]:
import asyncio


@tool
async def fetch_note_async(note_id: str) -> str:
    """Fetch a note asynchronously (simulated I/O delay)."""
    await asyncio.sleep(0.05)
    return NOTES.get(note_id, "not found")


async def async_tool_demo():
    result = await fetch_note_async.ainvoke({"note_id": "n2"})
    print("ainvoke:", result)


asyncio.run(async_tool_demo())

In FastAPI routes, use **`await llm_with_tools.ainvoke(...)`** and **`await tool.ainvoke(...)`** to avoid blocking the event loop (**07. Streaming, Batch, and Async**).

---

## 13. Tool Errors and ToolException

Tools should fail **predictably**. Raise **`ToolException`** for expected errors the model can recover from:

In [ ]:
from langchain_core.tools.base import ToolException


@tool
def get_note_strict(note_id: str) -> str:
    """Fetch a note by id; raises if id is invalid."""
    if note_id not in NOTES:
        raise ToolException(f"Unknown note id '{note_id}'. Valid ids: {list(NOTES)}")
    return NOTES[note_id]


try:
    get_note_strict.invoke({"note_id": "n99"})
except ToolException as exc:
    print("caught:", exc)

Return errors as **`ToolMessage`** content so the model can retry with corrected arguments — do not let raw stack traces reach the model in production.

---

## 14. Tools as Runnables

Every **`BaseTool`** supports the Runnable API — **`invoke`**, **`batch`**, **`ainvoke`**. Tools accept a **dict of args** or a full **tool call dict**:

In [ ]:
tc = {"name": "add", "args": {"a": 3, "b": 9}, "id": "demo-id", "type": "tool_call"}
print(add.invoke(tc["args"]))
print(add.batch([{"a": 1, "b": 2}, {"a": 10, "b": 5}]))

Because tools are Runnables, they can be composed in LCEL — though most apps invoke them from the tool loop rather than piping them into prompts directly.

---

## 15. RAG as a Tool

Wrap retrieval from **11. RAG with LCEL** as a tool so an agent can **choose** when to search docs vs call other APIs:

In [ ]:
@tool
def search_documentation(query: str) -> str:
    """Search internal documentation chunks. Use for product/API/security/testing questions."""
    return _search_notes(query)


doc_tools = [search_documentation, add]
doc_bound = llm.bind_tools(doc_tools)

rag_style = doc_bound.invoke(
    "Search documentation for JWT header format and explain briefly."
)

if rag_style.tool_calls:
    tc = rag_style.tool_calls[0]
    ctx = search_documentation.invoke(tc["args"])
    answer = llm.invoke([
        HumanMessage(content="Answer using ONLY this context:\n" + ctx),
        HumanMessage(content="What header carries JWT?"),
    ])
    print("retrieved:\n", ctx)
    print("--- answer ---")
    print(answer.content)

Pattern: **tool retrieves** → model **generates** in a follow-up. Agents combine both steps automatically. For always-on retrieval, use a RAG chain (**11**); for **optional** retrieval, expose search as a tool.

---

## 16. Tool Factories and Hidden Context

Tools should not expose secrets in their schema. Close over clients and credentials at construction time:

In [ ]:
def make_notes_tools(notes_db: dict[str, str]):
    @tool
    def get_note_local(note_id: str) -> str:
        """Fetch a note from the configured database."""
        return notes_db.get(note_id, "not found")

    return [get_note_local]


tenant_tools = make_notes_tools({"n1": "Tenant A configuration."})
print(tenant_tools[0].invoke({"note_id": "n1"}))

Each request/tenant can receive a **different tool list** with the same schema but different backing data — useful for multi-tenant agents (**16. Fallbacks and Configurable Runnables**).

---

## 17. Testing Tools Without Live API Calls

Test **tool logic** and **loop wiring** separately:

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

fake_ai = AIMessage(
    content="",
    tool_calls=[{
        "name": "add",
        "args": {"a": 2, "b": 3},
        "id": "call_test_1",
        "type": "tool_call",
    }],
)

tool_msgs = execute_tool_calls(fake_ai)
assert tool_msgs[0].content == "5"
assert tool_msgs[0].tool_call_id == "call_test_1"

fake_llm = FakeListChatModel(responses=["The sum is 5."])
follow_up = [
    HumanMessage(content="Add 2 and 3"),
    fake_ai,
    tool_msgs[0],
]
print(fake_llm.invoke(follow_up).content)
print("offline tool loop wiring passed")

Unit-test **`execute_tool_calls`** with fabricated **`AIMessage`** objects — no API key required. Integration-test the full loop with **`FakeListChatModel`** (**03. Chat Models and Messages**).

---

## 18. Tool Design Guidelines

| Guideline | Why |
|-----------|-----|
| **Small, focused tools** | Model picks correctly; easier to test |
| **Clear docstrings** | Descriptions drive routing |
| **Typed parameters** | Valid JSON args from the model |
| **Stable names** | `snake_case`; avoid renaming in production |
| **Idempotent reads** | Safe for retry loops |
| **Explicit errors** | `ToolException` or structured error strings |
| **No secrets in schema** | Use factories/closures for credentials |
| **Limit tool count** | >10–15 tools often hurts selection accuracy |

**Side-effect tools** (send email, charge card): add confirmation layers or human-in-the-loop (**13**, **17. Production Patterns for LangChain**).

---

## 19. Tools vs RAG vs Agents

| Pattern | When |
|---------|------|
| **RAG chain** (**11**) | Every query needs doc context |
| **Single tool + loop** | One API or calculator, few steps |
| **Multi-tool agent** (**13**) | Model chooses among search, SQL, code, APIs |

Tools are the **action layer**. RAG is one action among many.

---

## 20. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Missing docstring on `@tool` | `ValueError` at definition | Add docstring or `description=` |
| Expecting text when tools bound | Empty `content` | Check `tool_calls`; run loop |
| Wrong `tool_call_id` | Provider error | Match `ToolMessage` to `tc["id"]` |
| Partial parallel execution | Invalid conversation state | Execute **all** calls before next model turn |
| No iteration cap | Runaway cost | `max_steps` guard |
| Huge tool schemas | Wrong args, token bloat | Split tools; use Pydantic constraints |
| Secrets in tool parameters | Leaked to model/logs | Factory pattern with closed-over clients |
| Skipping tool unit tests | Broken executor in production | Test `execute_tool_calls` offline |

---

## 21. Summary

**Tools** expose typed, described functions to chat models. **`@tool`** is the default definition style; **`bind_tools`** attaches them to **`ChatOpenAI`**. The model returns **`AIMessage.tool_calls`**; your executor runs tools and replies with **`ToolMessage`** objects until the model produces a final text answer.

Key takeaways:

- Tool calling is a **message loop**, not a single invoke.
- **`description`** and **parameter schemas** determine routing quality.
- **`execute_tool_calls`** + **`max_steps`** form the manual agent skeleton.
- **Parallel tool calls** need one **`ToolMessage`** each.
- **Async tools** and **ToolException** cover I/O and recoverable errors.
- **RAG as a tool** lets agents choose when to retrieve.
- **Offline tests** fake `AIMessage.tool_calls` without API spend.

Demonstrations defined Notes API tools, bound them to the model, executed single and parallel calls, built a manual multi-step loop, wrapped search as a RAG-style tool, and tested wiring with fakes.

Next: **13. Agents with create_agent** — LangChain's built-in agent graph that automates the tool loop with middleware, memory hooks, and structured outputs.